**The Golden Rule First — "Patch Where It's Used"**

This is the #1 mistake junior developers make and the #1 thing interviewers test.

In [ ]:
# email_service.py
from smtplib import SMTP   # ← imported HERE, lives in THIS module's namespace

def send_welcome_email(to: str):
    smtp = SMTP("smtp.gmail.com")
    smtp.sendmail("app@example.com", to, "Welcome!")

In [ ]:
# ❌ WRONG — patching the SOURCE, not where it's used
@patch("smtplib.SMTP")
def test_wrong(mock_smtp): ...
# smtplib.SMTP is patched but email_service.SMTP is a different reference!
# The real SMTP still runs in email_service.py

# ✅ CORRECT — patch where the name LIVES after import
@patch("email_service.SMTP")
def test_correct(mock_smtp): ...
# email_service.SMTP is now the mock — this is what the function uses

**Memory model**:

After: `from smtplib import SMTP`

- `smtplib` module:      { `SMTP: <real class>` }
- `email_service` module:{ `SMTP: <real class>` }  ← a COPY of the reference

- `patch("smtplib.SMTP")`      replaces `smtplib.SMTP`     — **too late**, copy already made
- `patch("email_service.SMTP")` replaces `email_service.SMTP` — ✅ correct target

---

### `patch` as Decorator vs Context Manager — When to Use Which
```shell
@patch(...)          decorator    → use when entire test needs the mock
                                    mock injected as argument, auto-restored after test

with patch(...) as m: context mgr → use when only PART of test needs the mock
                                    fine-grained control of when mock is active

patch.object(...)    both         → patch an attribute ON a specific object/class
patch.dict(...)      both         → patch dictionary entries (os.environ, settings)
patch.multiple(...)  both         → patch many things at once
```

---

**1. `@patch` as a Decorator**

In [ ]:
# services/email_service.py
import requests
from smtplib import SMTP

def get_user_email(user_id: int) -> str:
    response = requests.get(f"https://api.example.com/users/{user_id}")
    response.raise_for_status()
    return response.json()["email"]

def send_welcome_email(to: str) -> bool:
    smtp = SMTP("smtp.gmail.com", 587)
    smtp.sendmail("app@example.com", to, "Welcome!")
    smtp.quit()
    return True

In [ ]:
# tests/test_email_service.py
import pytest
from unittest.mock import patch, MagicMock
from services.email_service import get_user_email, send_welcome_email


# ── Single patch ──────────────────────────────────────────────
@patch("services.email_service.requests.get")   # ← patched where used
def test_get_user_email(mock_get):
    # mock_get injected as LAST argument (bottom decorator = first arg)
    mock_get.return_value.json.return_value = {"email": "alice@example.com"}
    mock_get.return_value.raise_for_status.return_value = None

    result = get_user_email(1)

    assert result == "alice@example.com"
    mock_get.assert_called_once_with("https://api.example.com/users/1")


# ── Stacked patches — ORDER MATTERS ──────────────────────────
@patch("services.email_service.SMTP")           # ← decorator 2 (outer)
@patch("services.email_service.requests.get")   # ← decorator 1 (inner)
def test_stacked_patches(mock_get, mock_smtp):
    # Arguments go BOTTOM decorator → TOP decorator
    # inner (requests.get) → mock_get   (1st arg)
    # outer (SMTP)         → mock_smtp  (2nd arg)

    mock_get.return_value.json.return_value = {"email": "alice@example.com"}
    mock_get.return_value.raise_for_status.return_value = None

    mock_smtp_instance = MagicMock()
    mock_smtp.return_value = mock_smtp_instance  # SMTP(...) returns our mock

    email = get_user_email(1)
    result = send_welcome_email(email)

    assert result is True
    mock_smtp.assert_called_once_with("smtp.gmail.com", 587)
    mock_smtp_instance.sendmail.assert_called_once_with(
        "app@example.com", "alice@example.com", "Welcome!"
    )
    mock_smtp_instance.quit.assert_called_once()

**2. `patch` as a Context Manager**

In [ ]:
# services/order_service.py
import boto3                           # AWS SDK — real 3rd party
from django.conf import settings
from .email_service import send_welcome_email

def create_order(user_id: int, amount: float) -> dict:
    # Hits real AWS SQS in production
    sqs = boto3.client("sqs", region_name="eu-west-1")
    sqs.send_message(
        QueueUrl=settings.ORDER_QUEUE_URL,
        MessageBody=f"order:{user_id}:{amount}"
    )
    send_welcome_email(f"user_{user_id}@example.com")
    return {"user_id": user_id, "amount": amount, "status": "created"}

In [ ]:
# tests/test_order_service.py
from unittest.mock import patch, MagicMock
from services.order_service import create_order


def test_create_order_context_manager():
    with patch("services.order_service.boto3") as mock_boto3, \
         patch("services.order_service.send_welcome_email") as mock_email:
        # ↑ two context managers — comma-separated, no nesting needed

        # Configure the mock SQS client chain
        mock_sqs = MagicMock()
        mock_boto3.client.return_value = mock_sqs

        result = create_order(user_id=1, amount=99.99)

        # Assert result
        assert result == {"user_id": 1, "amount": 99.99, "status": "created"}

        # Assert SQS interaction
        mock_boto3.client.assert_called_once_with("sqs", region_name="eu-west-1")
        mock_sqs.send_message.assert_called_once_with(
            QueueUrl=settings.ORDER_QUEUE_URL,
            MessageBody="order:1:99.99"
        )

        # Assert email sent
        mock_email.assert_called_once_with("user_1@example.com")

    # ← mock is gone here — real boto3 restored automatically


def test_context_manager_scoping():
    """
    Shows WHY context manager beats decorator for partial mocking.
    Only the dangerous DB call is mocked — real logic runs normally.
    """
    order_data = {"user_id": 1, "amount": 99.99}

    # Step 1: real validation runs (no mock)
    assert order_data["amount"] > 0  # real logic

    # Step 2: only mock the external I/O
    with patch("services.order_service.boto3") as mock_boto3, \
         patch("services.order_service.send_welcome_email"):
        mock_boto3.client.return_value = MagicMock()
        result = create_order(**order_data)

    # Step 3: real assertions on result (no mock)
    assert result["status"] == "created"

**3. `patch.object` — Patch an Attribute on a Specific Instance or Class**

In [ ]:
# services/payment_service.py
import stripe   # 3rd party

class PaymentService:
    def __init__(self):
        self.stripe = stripe

    def charge(self, amount: int, token: str) -> dict:
        charge = self.stripe.Charge.create(
            amount=amount,
            currency="usd",
            source=token,
        )
        return {"charge_id": charge["id"], "status": charge["status"]}

In [ ]:
# tests/test_payment_service.py
import pytest
import stripe
from unittest.mock import patch, MagicMock
from services.payment_service import PaymentService


# ── patch.object as decorator ─────────────────────────────────
@patch.object(stripe.Charge, "create")
def test_charge_decorator(mock_create):
    mock_create.return_value = {"id": "ch_123", "status": "succeeded"}

    service = PaymentService()
    result  = service.charge(amount=1000, token="tok_visa")

    assert result == {"charge_id": "ch_123", "status": "succeeded"}
    mock_create.assert_called_once_with(
        amount=1000, currency="usd", source="tok_visa"
    )


# ── patch.object as context manager ──────────────────────────
def test_charge_context_manager():
    with patch.object(stripe.Charge, "create") as mock_create:
        mock_create.return_value = {"id": "ch_456", "status": "succeeded"}

        service = PaymentService()
        result  = service.charge(1000, "tok_visa")

        assert result["charge_id"] == "ch_456"


# ── patch.object on an INSTANCE ──────────────────────────────
def test_charge_on_instance():
    service = PaymentService()

    with patch.object(service, "stripe") as mock_stripe:
        mock_stripe.Charge.create.return_value = {
            "id": "ch_789", "status": "succeeded"
        }
        result = service.charge(1000, "tok_visa")
        assert result["charge_id"] == "ch_789"

**4. `patch.dict` — Patching Environment Variables and Settings**

In [ ]:
# config.py
import os

def get_database_url() -> str:
    url = os.environ.get("DATABASE_URL")
    if not url:
        raise ValueError("DATABASE_URL environment variable not set")
    return url

def get_feature_flags() -> dict:
    return {
        "new_checkout":  os.environ.get("FF_NEW_CHECKOUT", "false") == "true",
        "dark_mode":     os.environ.get("FF_DARK_MODE",    "false") == "true",
    }

In [ ]:
# tests/test_config.py
import os
import pytest
from unittest.mock import patch
from config import get_database_url, get_feature_flags


# ── patch.dict as decorator ───────────────────────────────────
@patch.dict(os.environ, {"DATABASE_URL": "postgresql://localhost/testdb"})
def test_get_database_url():
    assert get_database_url() == "postgresql://localhost/testdb"
    # os.environ restored to original after test


# ── patch.dict as context manager ────────────────────────────
def test_get_database_url_missing():
    with patch.dict(os.environ, {}, clear=True):
        # clear=True removes ALL existing env vars inside the block
        with pytest.raises(ValueError, match="DATABASE_URL"):
            get_database_url()


# ── Partial dict patch — only adds/overrides specified keys ──
@patch.dict(os.environ, {
    "FF_NEW_CHECKOUT": "true",
    "FF_DARK_MODE":    "false",
})
def test_feature_flags_checkout_on():
    flags = get_feature_flags()
    assert flags["new_checkout"] is True
    assert flags["dark_mode"]    is False


# ── Django settings patch ─────────────────────────────────────
from django.conf import settings as django_settings

def test_with_django_settings():
    with patch.dict(vars(django_settings), {"DEBUG": False, "ALLOWED_HOSTS": ["*"]}):
        assert django_settings.DEBUG is False

**5. Real FastAPI Test — Patching Your Dependencies**

In [ ]:
# tests/test_routes.py
import pytest
from unittest.mock import patch, AsyncMock
from fastapi.testclient import TestClient
from main import app


client = TestClient(app)


@pytest.mark.parametrize("user_groups, endpoint, expected_status", [
    pytest.param(["admin"],     "/api/posts/1", 200, id="admin-success"),
    pytest.param(["viewer"],    "/api/posts/1", 403, id="viewer-forbidden"),
    pytest.param(["moderator"], "/api/posts/1", 403, id="moderator-forbidden"),
])
def test_delete_post_rbac(user_groups, endpoint, expected_status):
    mock_user = {"id": 1, "username": "testuser", "groups": user_groups}

    # Patch get_current_user in the module where it's USED (dependencies.py)
    with patch("dependencies.get_current_user", return_value=mock_user):
        response = client.delete(endpoint)

    assert response.status_code == expected_status


# ── AsyncMock for async functions ─────────────────────────────
@pytest.mark.asyncio
async def test_async_service():
    """
    AsyncMock is required for async functions.
    Regular MagicMock won't work — it returns a coroutine, not a mock.
    """
    with patch(
        "services.user_service.fetch_from_db",
        new_callable=AsyncMock
    ) as mock_fetch:
        mock_fetch.return_value = {"id": 1, "name": "Alice"}

        from services.user_service import get_user
        result = await get_user(1)

        assert result["name"] == "Alice"
        mock_fetch.assert_awaited_once_with(1)  # ← assert_awaited not assert_called!

**6. Mock Assertion Cheatsheet**

```shell
mock = MagicMock()
mock("a", key="b")

# ── Call assertions ───────────────────────────────────────────
mock.assert_called()                        # called at least once
mock.assert_called_once()                   # called exactly once
mock.assert_called_with("a", key="b")       # last call had these args
mock.assert_called_once_with("a", key="b")  # called once with these args
mock.assert_not_called()                    # never called

# ── Inspection ────────────────────────────────────────────────
mock.call_count                     # int — how many times called
mock.call_args                      # args of the LAST call
mock.call_args_list                 # list of ALL calls
mock.called                         # bool

# ── AsyncMock specific ────────────────────────────────────────
async_mock.assert_awaited()
async_mock.assert_awaited_once_with(1)
async_mock.await_count

# ── Chained calls ─────────────────────────────────────────────
mock_boto3.client.return_value.send_message.assert_called_once()
#          ↑ boto3   ↑ .client()            ↑ .send_message()
```

---

**The Complete Mental Model**
```shell
WHAT to patch:
  ├── A function used inside your code      → patch("mymodule.func_name")
  ├── A class method                        → patch.object(MyClass, "method")
  ├── An instance attribute                 → patch.object(instance, "attr")
  └── A dict / env variable                 → patch.dict(os.environ, {...})

WHERE to patch:
  └── Always the module that IMPORTS it, not where it's defined
      from smtplib import SMTP  →  patch("mymodule.SMTP")  ✅
                                    patch("smtplib.SMTP")  ❌

HOW to patch:
  ├── @patch     → whole test needs mock, injected as last args (reversed order)
  ├── with patch → partial test needs mock, use when timing matters
  └── both can be combined freely

ASYNC:
  └── async functions → new_callable=AsyncMock, assert_awaited_once_with()
```

The interview answer: "The rule I always follow is patch where the name is used, not where it's defined. I use the decorator form when the entire test depends on the mock, and the context manager when I need fine-grained control — like asserting real logic before and after the patched section. For environment and settings I use patch.dict, and for async code I switch to AsyncMock with assert_awaited instead of assert_called."